In [ ]:
import time
import numpy as np
# import openmdao.api as om
from scipy.spatial import ConvexHull
# import sys
import pandas as pd
# from py_wake import NOJ
# from py_wake.deflection_models import JimenezWakeDeflection
# from py_wake.turbulence_models.stf import STF2017TurbulenceModel
# from py_wake.deficit_models.gaussian import ZongGaussianDeficit
# from py_wake.superposition_models import WeightedSum, SqrMaxSum, LinearSum
# from py_wake.rotor_avg_models import GaussianOverlapAvgModel, CGIRotorAvg
# from py_wake.wind_farm_models import PropagateDownwind
# from py_wake.turbulence_models import CrespoHernandez
# from py_wake.deflection_models.jimenez import JimenezWakeDeflection
# from py_wake.site._site import UniformSite
# from py_wake.deficit_models.utils import ct2a_mom1d


from topfarm.utils import regular_generic_layout

from hydesign.examples import examples_filepath
from hydesign.assembly.hpp_pywake_gnn_loads import hpp_model

from design_friendly.utils.iea22s import IEA22s
# from design_friendly.utils.sites import Hornsrev1Site
from design_friendly.utils.easy import easy_yaw_gnn
from design_friendly.utils.get_flowmodel import get_flowmodel
# import xarray as xr
from py_wake.examples.data.dtu10mw_surrogate import DTU10MW_1WT_Surrogate


In [ ]:

start = time.time()
loc = 0  # 0: Denmark, 1: France, 2: Germany
sample = 0  # sample index
names = ["Denmark_good_wind", 'France_good_wind', 'Germany_good_wind']
name = names[loc]
examples_sites = pd.read_csv(
    f"{examples_filepath}examples_sites.csv", index_col=0, sep=";"
)
ex_site = examples_sites.loc[examples_sites.name == name]

longitude = ex_site["longitude"].values[0]
latitude = ex_site["latitude"].values[0]
altitude = ex_site["altitude"].values[0]

sim_pars_fn = examples_filepath + ex_site["sim_pars_fn"].values[0]
input_ts_fn = examples_filepath + ex_site["input_ts_fn"].values[0]

# this is a smoother version of PyWake IEA22 that works better with wake steering optimization
wt = IEA22s()
wds = np.arange(0, 360, 2)
wss = np.arange(3, 25, 1)  # we don't really need all this range
TI = 0.06  # site.local_wind().TI_ilk.ravel()

life_y = 25
intervals_per_hour = 1
n_wt = 5
print('n_wt:', n_wt)
print('')
d = 284
hh = 170
RP = 22.0
G_MW = int(n_wt * RP * 1.0)  # grid connection size
sx = 4 * d
sy = 5 * d
x, y = regular_generic_layout(n_wt, sx, sy, stagger=0, rotation=0)

hull = ConvexHull(list(np.asarray([x, y]).T))
area = hull.volume

N_ws = 365 * 24 * intervals_per_hour
time_stamp = np.arange(N_ws) / 6 / 24
farm = get_flowmodel(wt=wt)

yaws = easy_yaw_gnn(x, y, wd=wds, ws=wss, TI=TI)
yaw = yaws
tilt = np.zeros(N_ws)

wind_MW_per_km2=n_wt * RP / (area * 10**-6)
b_P = 50
wt_load = DTU10MW_1WT_Surrogate()
farm_load = get_flowmodel(wt=wt_load)

hpp = hpp_model(
    n_wt=n_wt,
    farm=farm,
    farm_load=farm_load,
    latitude=latitude,
    longitude=longitude,
    altitude=altitude,
    sim_pars_fn=sim_pars_fn,
    input_ts_fn=input_ts_fn,
    intervals_per_hour=intervals_per_hour,
    x=x,
    y=y,
    tilt=tilt,
    time_stamp=time_stamp,
    G_MW=G_MW,
    TI=TI,
    CAPEX_ref = 113,  #MEURO
)

# Wind plant design
x = dict(
    clearance = hh - d / 2,
    sp=RP *10 ** 6 / (np.pi * d ** 2 / 4),
    p_rated=RP,
    Nwt=n_wt,
    wind_MW_per_km2=wind_MW_per_km2,
    # PV plant design
    solar_MW=0,
    surface_tilt=28.125,
    surface_azimuth=191.250,
    DC_AC_ratio=1.479,
    # Energy storage & EMS price constrains
    b_P=b_P,
    b_E_h=4,
    cost_of_battery_P_fluct_in_peak_price_ratio=8.750,
    # Wind turbine control
    yaw=yaw,
)
end = time.time()
print("GNN time [min]:", (end - start) / 60)



In [ ]:

start = time.time()
outs = hpp.evaluate(**x)

hpp.print_design(list(x.values()), outs)

end = time.time()
print("HPP time [min]:", (end - start) / 60)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(hpp.prob['M'])

In [ ]:
n = 300
start = 200
end = n + start
plt.plot(hpp.prob['lambda_t'][start:end])

In [ ]:

plt.plot(hpp.prob['wind_t'][start:end], label='wind_t')
plt.plot(hpp.prob['wind_t_ref'][start:end], label='wind_t_ref')
# plt.plot(hpp.prob['wind_t_actual'][start:end], label='wind_t_actual')
plt.legend()

In [ ]:
n_days = 4
start_day = 34
start = start_day * 24
end = n_days * 24 + start
plt.plot(((hpp.prob['wind_t']-hpp.prob['wind_t_ref'])*hpp.prob['price_t'])[start:end], label='d_wind_t * price')
# plt.plot((hpp.prob['wind_t_ref']*hpp.prob['price_t'])[start:end], label='wind_t_ref')
plt.plot(hpp.prob['M'][start:end], label='M')
# plt.plot(hpp.prob['wind_t_actual'][start:end], label='wind_t_actual')
plt.legend()


In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8))
d_wind = hpp.prob['wind_t'] - hpp.prob['wind_t_ref']
lamb = hpp.prob['lambda_t']
M = hpp.prob['M']
price = hpp.prob['price_t']
hpp_curt_t = hpp.prob['hpp_curt_t']

b_t = hpp.prob['b_t'][:8760]

# long term signals
hpp_t = hpp.prob['hpp_t']
price_t_ext = hpp.prob['price_t_ext']


x_axis = np.arange(start, end)

# middle: price-weighted delta and M
axes[0].plot(x_axis, (d_wind * price)[start:end],
             label='rev. from increased power prod.', alpha=0.8)
axes[0].plot(x_axis, M[start:end], label='increased O&M costs from using wake steering', alpha=0.8)
# axes[0].plot((b_t * price)[start:end], label='value from battery charge/discharge', alpha=0.8)
axes[0].set_ylabel('Value [€]')
axes[0].legend(loc='best')
axes[0].grid(True, which='both', linestyle='--', alpha=0.4)

# bottom: lambda
axes[1].plot(x_axis, lamb[start:end])
axes[1].set_ylabel('Lambda')
axes[1].set_xlabel('Time step [h]')
axes[1].legend(loc='best')
axes[1].grid(True, which='both', linestyle='--', alpha=0.4)

# top: wind signals
axes[2].plot(x_axis, (d_wind * price * lamb - M * lamb)[start:end], label='increase in rev. smart yaw', linewidth=5)
axes[2].plot(x_axis, (d_wind * price - M )[start:end], label='increase in rev. optimal yaw', linestyle='--')
axes[2].plot(x_axis, (np.zeros_like(d_wind))[start:end], label='increase in rev. no yaw')

# axes[2].plot(hpp.prob['wind_t_ref'][start:end], label='wind_t_ref', linestyle='--')
axes[2].set_ylabel('revenue increase [€]')
axes[2].legend(loc='best')
axes[2].grid(True, which='both', linestyle='--', alpha=0.4)

fig.tight_layout()
# ...existing code...

In [ ]:
print('total increase in revenue smart yaw:', np.sum((d_wind * price * lamb)))
print('total increase in O&M costs smart yaw:', np.sum(M * lamb))

from hydesign.ems.ems import expand_to_lifetime
d_wind_ext = expand_to_lifetime(d_wind, life_y=25)
lamb_ext = expand_to_lifetime(lamb, life_y=25)
M_ext = expand_to_lifetime(M, life_y=25)

print('total relative increase in revenue smart yaw:', (np.sum((d_wind_ext * price_t_ext * lamb_ext))-np.sum(M_ext * lamb_ext))/np.sum(hpp_t*price_t_ext))
# print('total relative ncrease in O&M costs smart yaw:', np.sum(M * lamb))

In [ ]:
np.sum(hpp_t*price_t_ext)

In [ ]:
np.sum((d_wind_ext * price_t_ext * lamb_ext))-np.sum(M_ext * lamb_ext)

In [ ]:
(395821711+ 4372382)/395821711

In [ ]:
M_assumed = hpp.prob['M'].sum()
M_actual = hpp.prob['M_actual']
print('increase in M from yaw control:', (M_actual - M_assumed)/M_assumed * 100, '%')

In [ ]:
M_assumed

In [ ]:
M_actual

In [ ]:
hpp.prob['M']

In [ ]:
load_sensors = ['Blade_root_edgewise_M_y', 'Blade_root_flapwise_M_x', 'Tower_top_tilt_M_x', 'Tower_top_yaw_M_z']
load_sensors_used = ['Blade_root_flapwise_M_x','Blade_root_flapwise_M_x', 'Blade_root_flapwise_M_x', 'Tower_top_yaw_M_z']

sensors_to_plot = [1, 3]
DEL = hpp.prob['DEL']
DEL_ref = hpp.prob['DEL_ref']
DEL_actual = hpp.prob['DEL_actual']
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 8))
for idx, i in enumerate(sensors_to_plot):
    axes[idx].plot(x_axis, DEL[i, 0, :][start:end], label='DEL control', linewidth=5)
    axes[idx].plot(x_axis, DEL_ref[i, 0, :][start:end], label='DEL_ref')
    axes[idx].plot(x_axis, DEL_actual[i, 0, :][start:end], label='DEL_actual' ,linestyle='--')
    axes[idx].set_ylabel('DEL sensor ' + str(load_sensors[i]) + ' [kNm]')
    axes[idx].legend(loc='best')
    axes[idx].grid(True, which='both', linestyle='--', alpha=0.4)


In [ ]:
np.unravel_index(np.argmax(DEL_actual - DEL), DEL_actual.shape)

In [ ]:
6179/24


In [ ]:
DEL.shape

In [ ]:
wholer_exponents = hpp.prob['wholer_exponents']
LDEL_control = (((DEL/DEL.mean()) ** wholer_exponents[:, np.newaxis, np.newaxis]).sum((2))*3600*24*365*20/10**7) ** (1 / wholer_exponents[:, np.newaxis]) * DEL.mean()
LDEL_ref = (((DEL_ref/DEL_ref.mean()) ** wholer_exponents[:, np.newaxis, np.newaxis]).sum((2))*3600*24*365*20/10**7) ** (1 / wholer_exponents[:, np.newaxis]) * DEL_ref.mean()
LDEL_actual = (((DEL_actual/DEL_actual.mean()) ** wholer_exponents[:, np.newaxis, np.newaxis]).sum((2))*3600*24*365*20/10**7) ** (1 / wholer_exponents[:, np.newaxis]) * DEL_actual.mean()

In [ ]:
print(LDEL_control, LDEL_ref, LDEL_actual)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Recompute LDEL arrays if they don't exist (safe guard)

Lc = LDEL_control
Lr = LDEL_ref
La = LDEL_actual

# normalize by mean LDEL_ref across turbines for each signal
mean_ref_per_signal = Lr.mean(axis=1, keepdims=True)  # shape (n_signals, 1)
# avoid division by zero
mean_ref_per_signal[mean_ref_per_signal == 0] = 1.0

Lc_norm = Lc / mean_ref_per_signal
Lr_norm = Lr / mean_ref_per_signal
La_norm = La / mean_ref_per_signal

_, n_turbines = Lc_norm.shape
n_signals = 2
turbines = np.arange(n_turbines)

fig, axes = plt.subplots(n_signals, 1, sharex=True, figsize=(10, 2.5 * max(1, n_signals)))


width = 0.25
for i, ax in enumerate(axes):
    x = turbines
    ax.bar(x - width, Lc_norm[i, :], width=width, label='LDEL control (norm)', alpha=0.9)
    ax.bar(x,       Lr_norm[i, :], width=width, label='LDEL ref (norm)', alpha=0.8)
    ax.bar(x + width, La_norm[i, :], width=width, label='LDEL actual (norm)', alpha=0.8)
    ax.set_ylabel(f'Normalized LDEL \n {load_sensors[sensors_to_plot[i]]}')
    ax.set_xticks(turbines)
    ax.set_xticklabels(turbines)
    ax.grid(True, which='both', linestyle='--', alpha=0.4)
    ax.legend(loc='best')

axes[-1].set_xlabel('Turbine index')
fig.tight_layout()
plt.show()